# Proyecto P02: Aprendizaje supervisado — Guía del ejercicio

En este proyecto vas a construir tus primeros clasificadores con scikit-learn: un k-vecinos más cercanos (k-NN) para predecir el riesgo de ataque al corazón, otro para reconocer dígitos escritos a mano (incluido uno que dibujes tú) y una regresión logística para compararlos. También practicarás la división en entrenamiento y prueba y la elección de $k$ sin caer en sobreajuste ni subajuste.

Cuando termines, compara tu trabajo con la solución: [p02-aprendizaje-supervisado-solucion.ipynb](p02-aprendizaje-supervisado-solucion.ipynb).

## Contexto

Un hospital quiere una herramienta que, a partir de los resultados clínicos de un paciente, indique bajo qué escenario es más probable que sufra un **ataque al corazón**. Un cardiólogo puede hacer esa evaluación sin *machine learning*, pero no al instante, y desde luego no para cientos o miles de pacientes. Tu primera tarea es explorar qué factores se asocian con el riesgo y entrenar un clasificador.

En la segunda parte cambiarás de problema para practicar la clasificación con muchas clases: reconocer los dígitos del 0 al 9 escritos a mano en el conjunto MNIST, medir la precisión del modelo con datos que no ha visto y probarlo con una imagen dibujada por ti.

## Datos

Los archivos están en esta misma carpeta.

**`heart-attack.csv`**: 303 pacientes, uno por fila, sin valores faltantes.

| Columna | Descripción |
| --- | --- |
| `age` | Edad del paciente |
| `sex` | Sexo del paciente (0/1) |
| `cp` | Tipo de dolor de pecho: 0 = angina típica, 1 = angina atípica, 2 = dolor no anginoso, 3 = asintomático |
| `trtbps` | Presión arterial en reposo (mm Hg) |
| `chol` | Colesterol (mg/dl) |
| `fbs` | Glucosa en ayunas > 120 mg/dl: 1 = sí, 0 = no |
| `restecg` | Electrocardiograma en reposo: 0 = normal, 1 = anomalía de la onda ST-T, 2 = hipertrofia ventricular izquierda |
| `thalachh` | Frecuencia cardiaca máxima alcanzada |
| `exng` | Angina inducida por el ejercicio: 1 = sí, 0 = no |
| `oldpeak` | Depresión del segmento ST (pico previo) |
| `slp` | Pendiente del segmento ST |
| `caa` | Número de vasos principales (0–4) |
| `thall` | Resultado de la prueba de esfuerzo con talio (0–3) |
| `output` | Variable objetivo: 1 = mayor probabilidad de ataque al corazón, 0 = menor |

**`mnist.csv`**: 42 000 imágenes de dígitos escritos a mano (unos 76 MB). Cada fila es una imagen de 28 × 28 píxeles: la columna `label` es el dígito (0–9) y las columnas `pixel0` a `pixel783` son los 784 píxeles, con valores de 0 (fondo) a 255 (trazo). Por su tamaño, puedes hacer pruebas rápidas leyendo solo una parte, por ejemplo con `pd.read_csv("mnist.csv", nrows=5000)`.

**`imagen-muestra.png`**: un dígito de ejemplo dibujado en blanco sobre fondo negro, por si no quieres dibujar el tuyo.

## Instrucciones

- Trabaja en este notebook, ejecutándolo desde su carpeta para que las rutas relativas funcionen (`uv run jupyter lab` desde la raíz del repositorio).
- Usa scikit-learn para los modelos, matplotlib para las imágenes y Pillow (`PIL`) para procesar tu dibujo.
- Escribe tu código en las celdas vacías que siguen a cada tarea y responde por escrito las preguntas.

La celda siguiente importa las librerías y carga los datos de ataques al corazón; `mnist.csv` se carga en la Tarea 4.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("heart-attack.csv")
df.info()

## Tarea 1. Análisis exploratorio

¿Cuáles son los factores que pueden aumentar o disminuir la probabilidad de un ataque al corazón?

1. Comprueba que no hay valores nulos.
2. Prepara las variables para el modelo: codifica las categóricas (`sex`, `cp`, `fbs`, `restecg`, `slp`, `caa`, `thall`, `exng`) con *one-hot* (eliminando la primera categoría) y discretiza las numéricas (`age`, `trtbps`, `chol`, `thalachh`, `oldpeak`) en 5 intervalos de igual ancho. Puedes combinar `OneHotEncoder` y `KBinsDiscretizer` en un `ColumnTransformer`.
3. Revisa cuántos pacientes hay en cada clase de `output`.
4. Calcula la correlación de cada variable con `output` y ordénalas de mayor a menor.

*Opcional:* la librería `dataprep` (`from dataprep.eda import create_report`) genera un reporte exploratorio automático, aunque su instalación puede fallar en algunos sistemas.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 2. Clasificador k-NN

Es hora de crear tu primer clasificador. scikit-learn necesita los datos en una matriz en la que cada columna sea una variable y cada fila una observación (aquí, el registro clínico de un paciente), y la variable objetivo en una sola columna con el mismo número de observaciones.

1. Crea `X` con las variables independientes e `y` con `output`.
2. Crea un clasificador `KNeighborsClassifier` con **6 vecinos** y ajústalo a los datos.

In [ ]:
# Tu código aquí

## Tarea 3. Predecir una observación nueva

Todos los datos se usaron para entrenar, así que no quedan pacientes sin clasificar. Simula una observación completamente nueva (por ejemplo, con la mediana de cada variable), dale la forma de una matriz de una fila y predice su clase con `.predict()`.

In [ ]:
# Tu código aquí

## Tarea 4. Reconocimiento de dígitos

Hasta ahora la clasificación era binaria. Con MNIST hay 10 clases, los dígitos del 0 al 9.

1. Carga `mnist.csv` y revisa sus columnas.
2. Crea una lista `cols` con todas las columnas que contienen la palabra `pixel`.
3. Elige una imagen (por ejemplo, la primera), imprime su etiqueta (`label`) y muéstrala con `plt.imshow`, reacomodando sus 784 píxeles en una matriz de 28 × 28 (`cmap=plt.cm.gray_r`).

In [ ]:
# Tu código aquí

## Tarea 5. Entrenamiento y prueba

Una diferencia clave entre la estadística clásica y el *machine learning* es medir la precisión con datos que el modelo **no ha visto**. El objetivo es predecir qué dígito hay en cada imagen.

1. Crea `X` (los píxeles) e `y` (`label`).
2. Divide los datos en entrenamiento y prueba en proporción **70/30** con `train_test_split` (fija `random_state` para que sea reproducible).
3. Entrena un k-NN con **14 vecinos**, predice el conjunto de prueba y calcula la precisión con `accuracy_score`. ¿Cómo crees que se calcula la precisión de un modelo de clasificación?

In [ ]:
# Tu código aquí

## Tarea 6. Reconocer tu propia imagen

1. Dibuja un dígito con Paint u otro programa, en **blanco sobre fondo negro**, y guárdalo en formato PNG o JPG en esta carpeta (o usa `imagen-muestra.png`). Muéstralo con `plt.imshow`.
2. Con Pillow, redimensiona la imagen a 28 × 28 píxeles y conviértela en un arreglo de NumPy.
3. Adapta el arreglo al formato del modelo: quédate con un solo canal de color y aplánalo en una fila de 784 valores.
4. Predice el dígito con el k-NN de la Tarea 5 y muestra la imagen procesada. ¿Fue correcta la clasificación?

In [ ]:
# Tu código aquí

## Tarea 7. Sobreajuste y subajuste

¿Cuál es el número ideal de vecinos $k$? Para $k = 1, \dots, 8$, entrena un k-NN y calcula su precisión en el conjunto de entrenamiento y en el de prueba. Grafica ambas curvas contra $k$ (título, ejes y leyenda) y elige el valor de $k$ que evita tanto el sobreajuste como el subajuste.

In [ ]:
# Tu código aquí

## Tarea 8. Regresión logística

Haz la predicción de tu imagen, pero esta vez con una **regresión logística**. Para comparar los dos modelos con justicia, ajusta los hiperparámetros de ambos con `GridSearchCV` (validación cruzada de 5 particiones):

- k-NN: `n_neighbors` de 1 a 14, `weights` (`"uniform"`, `"distance"`) y `metric` (`"euclidean"`, `"manhattan"`).
- Regresión logística: `C` (0.1, 1, 10, 100), `solver` (`"liblinear"`, `"lbfgs"`) y `max_iter` (100, 200, 300).

Reporta los mejores parámetros, la precisión de validación cruzada y la precisión en prueba de cada modelo. ¿Cuál de los dos da mejores resultados? Con las 42 000 imágenes la búsqueda es lenta: si hace falta, usa una muestra (por ejemplo, 5 000 imágenes).

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Antes de terminar

- [ ] El notebook se ejecuta de principio a fin sin errores.
- [ ] La precisión del modelo de dígitos se midió en el conjunto de prueba.
- [ ] Probaste el modelo con una imagen propia de 28 × 28 píxeles.
- [ ] La gráfica de sobreajuste y subajuste muestra las curvas de entrenamiento y de prueba.
- [ ] Comparaste k-NN y regresión logística y respondiste cuál funciona mejor.

Cuando termines, compara tu trabajo con la solución: [p02-aprendizaje-supervisado-solucion.ipynb](p02-aprendizaje-supervisado-solucion.ipynb).